In [ ]:
#| hide
from cogitarelink.memory import *
from cogitarelink.retriever import *
from cogitarelink.vocabtools import *

# Installation

```bash
pip install cogitarelink
```

## Dependencies

Cogitarelink requires the following key dependencies:

- pyld: JSON-LD processing
- rdflib: RDF data manipulation
- claudette: Claude API integration
- fastcore: Core utilities

If you encounter import errors, ensure all dependencies are installed:

```bash
pip install pyld rdflib 'claudette>=0.2.0' fastcore httpx
```

# Overview

Cogitarelink is a Python library that provides semantic memory, linked data navigation, and vocabulary tools for AI agents, with built-in Claude integration. It enables agents to navigate and understand the [linked open data cloud](https://lod-cloud.net/), build structured knowledge representations, and work with JSON-LD contexts to organize and interpret data.

### Key Features

- **Semantic Memory**: Store, query, and manage semantic data with JSON-LD context awareness
- **LOD Navigation**: Browse and retrieve data from Linked Open Data sources
- **Vocabulary Tools**: Work with standard and custom vocabularies in JSON-LD
- **Claude Integration**: Tools for connecting Claude to semantic data structures

## Quick Start

```python
from cogitarelink.memory import SemanticMemory
from cogitarelink.retriever import LODNavigator
from cogitarelink.memorytools import query_memory_with_claude

# Create a semantic memory and navigator
memory = SemanticMemory()
navigator = LODNavigator()

# Load a vocabulary (e.g., Dublin Core)
memory.cache.preload_vocabulary("dc")

# Add some data to memory
document = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/document/123",
    "@type": "CreativeWork",
    "name": "Example Document",
    "creator": {"@id": "http://example.org/person/alice"}
}
memory.add(document)

# Query the memory with Claude
result = query_memory_with_claude(
    memory=memory,
    query="What creative works are in the memory and who created them?"
)
```

## Example: Working with Semantic Memory

In [ ]:
from cogitarelink.memory import SemanticMemory

# Create a semantic memory
memory = SemanticMemory()

# Add JSON-LD data to memory
person = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/person/alice",
    "@type": "Person",
    "name": "Alice Smith",
    "jobTitle": "Software Engineer",
    "knows": {"@id": "http://example.org/person/bob"}
}

memory.add(person)

# Query the memory
alice = memory.query_by_id("http://example.org/person/alice")
print(f"Found person: {alice.get('http://schema.org/name')}")

# Find entities by type
people = memory.query_by_type("http://schema.org/Person")
print(f"Found {len(people)} people in memory")

[GRAPH] GraphManager initialized
Found person: [{'@value': 'Alice Smith'}]
Found 1 people in memory


## Example: Navigating Linked Data

The LODNavigator class requires DSPy to be configured with a valid LM (Language Model) before use. This is because it uses Claude or other language models to analyze content and determine the best extraction strategy for linked data.

> **Note:** You'll need to set up your API key for the chosen language model. For Claude, set the `ANTHROPIC_API_KEY` environment variable.

In [ ]:
from cogitarelink.retriever import LODNavigator
import dspy

# Initialize DSPy with Claude as the Language Model (LM)
# Note: You must set an API key through environment variables or configuration
#import dspy # Add dspy import

# Configure DSPy LM for this notebook's context
try:
    # Use the same model as in 01_retriever.ipynb
    lm = dspy.LM('anthropic/claude-3-5-sonnet-20241022')
    dspy.configure(lm=lm)
    print("DSPy configured successfully.")
except Exception as e:
    print(f"Error configuring DSPy: {e}")
    # Optionally raise the error or handle it if configuration is critical
    # raise e

# Create a navigator
navigator = LODNavigator()

# Navigate to a Wikidata entity (e.g., Douglas Adams, Q42)
result = navigator.navigate("http://www.wikidata.org/entity/Q42")

if result["success"]:
    # Extract data from the JSON-LD
    jsonld = result["json_ld"]
    
    # Find the main entity in the graph
    main_entity = None
    if "@graph" in jsonld:
        for entity in jsonld["@graph"]:
            if entity.get("@id") == "http://www.wikidata.org/entity/Q42":
                main_entity = entity
                break
    
    if main_entity:
        # Extract a label if available
        label_key = "http://www.w3.org/2000/01/rdf-schema#label"
        if label_key in main_entity:
            for label in main_entity[label_key]:
                if isinstance(label, dict) and label.get("@language") == "en":
                    print(f"Found entity: {label.get('@value')}")
                    break

DSPy configured successfully.
Found entity: Douglas Adams


## Example: Working with Vocabularies

In [ ]:
from cogitarelink.vocabtools import VOCABULARY_REGISTRY, create_vocab_aware_document_loader
from pyld import jsonld

# Create a vocabulary-aware document loader
loader = create_vocab_aware_document_loader()
jsonld.set_document_loader(loader)

# List available vocabularies in the registry
for vocab_name, vocab_info in VOCABULARY_REGISTRY.items():
    print(f"Vocabulary: {vocab_name}")
    print(f"  URI: {vocab_info['uri']}")
    print(f"  Description: {vocab_info['description']}")
    print()

Vocabulary: schema
  URI: https://schema.org/
  Description: Schema.org vocabulary for structured data on the internet

Vocabulary: croissant
  URI: http://mlcommons.org/croissant/
  Description: Community Resource for Open and Inclusive Scholarly Sources and Artifacts for Novel Technologies

Vocabulary: ro-crate
  URI: https://w3id.org/ro/crate/1.2-DRAFT/context
  Description: Research Object Crate (RO-Crate) metadata specification

Vocabulary: vc
  URI: https://www.w3.org/ns/credentials/v2
  Description: W3C Verifiable Credentials data model and representations

Vocabulary: epcis
  URI: https://ref.gs1.org/epcis/
  Description: GS1 EPCIS standard for supply chain visibility

Vocabulary: dc
  URI: http://purl.org/dc/terms/
  Description: Dublin Core Metadata Initiative Terms

Vocabulary: foaf
  URI: http://xmlns.com/foaf/0.1/
  Description: Friend of a Friend vocabulary for describing people and their connections

Vocabulary: dcat
  URI: http://www.w3.org/ns/dcat#
  Description: W3C D

## Example: Using Claude with Semantic Memory

In [ ]:
from cogitarelink.memory import SemanticMemory
from cogitarelink.memorytools import create_entity_finder_tools
from claudette import Chat

# Set up a memory with some test data
memory = SemanticMemory()

# Add some test data
products = [
    {
        "@context": {"@vocab": "http://schema.org/"},
        "@id": "http://example.org/product/1",
        "@type": "Product",
        "name": "Smartphone Pro",
        "description": "Latest smartphone with advanced features",
        "brand": {"@id": "http://example.org/brand/techco"}
    },
    {
        "@context": {"@vocab": "http://schema.org/"},
        "@id": "http://example.org/brand/techco",
        "@type": "Brand",
        "name": "TechCo",
        "slogan": "Innovation for everyone"
    }
]

for product in products:
    memory.add(product)

# Create Claude-compatible tools for working with memory
tools = create_entity_finder_tools(memory)

# Create a chat with these tools
chat = Chat(model="claude-3-5-sonnet-20240620", tools=list(tools.values()))

# Create a prompt that asks Claude to explore the memory
prompt = """
Please explore the semantic memory to answer these questions:
1. What types of entities are in the memory?
2. What product(s) are in the memory and what are their details?
3. How are the products related to brands?

Use the available tools to search the memory and provide a summary of what you find.
"""

# Use toolloop to let Claude use the tools to explore memory
response = chat.toolloop(prompt)

ImportError: cannot import name 'create_entity_finder_tools' from 'cogitarelink.memorytools' (/Users/cvardema/dev/git/LA3D/cogitarelink/cogitarelink/memorytools.py)

from cogitarelink.memory import SemanticMemory
from cogitarelink.vocabtools import register_vocab_aware_loader

# Register a vocabulary-aware document loader
register_vocab_aware_loader()

# Create memory with context handling
memory = SemanticMemory()

# Define a custom context
context = {
    "@context": {
        "@version": 1.1,
        "@vocab": "http://schema.org/",
        "person": "Person",
        "name": "name",
        "knows": {"@id": "knows", "@type": "@id"}
    }
}

# Use the context with an entity
person = {
    "@context": context["@context"],
    "@id": "http://example.org/person/alice",
    "@type": "Person",
    "name": "Alice Smith",
    "knows": {"@id": "http://example.org/person/bob"}
}

# Register the context in memory
memory.add(person)

# Query for the entity
alice = memory.query_by_id("http://example.org/person/alice")
print(f"Found entity: {alice['@id']}")

## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install cogitarelink in Development mode

```sh
# make sure cogitarelink package is installed in development mode
$ uv venv
$ uv pip install -e ".[dev]"

# make changes under nbs/ directory
# ...

# compile to have changes apply to cogitarelink
$ nbdev_prepare
```

## Installation

Install latest from the GitHub [repository](https://github.com/la3d/cogitarelink):

```sh
$ pip install -U git+https://github.com/la3d/cogitarelink.git
```

Or install from PyPI:

```sh
$ pip install cogitarelink
```

### Documentation

Documentation can be found hosted on this GitHub [repository](https://github.com/la3d/cogitarelink)'s [pages](https://la3d.github.io/cogitarelink/).

## Contributing

We welcome contributions! Please see our [contributing guide](https://github.com/la3d/cogitarelink/blob/main/CONTRIBUTING.md) for details.

## License

This project is licensed under the MIT License - see the [LICENSE](https://github.com/la3d/cogitarelink/blob/main/LICENSE) file for details.